In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

df = pd.read_csv('simpler_data.csv')

MIN_WAVELENGTH = 900
MAX_WAVELENGTH = 1690
FEATURE_COLS   = [col for col in df.columns if
                  col.isdigit() and MIN_WAVELENGTH <= int(col) <= MAX_WAVELENGTH]
TARGET_COLS    = ['gv_fraction', 'npv_fraction', 'soil_fraction']

RANDOM_STATE     = 42
N_SAMPLES        = 144
TEST_SIZE        = 0.2
CORR_THRESHOLD   = 0.90

df_subset = df.sample(n=N_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)
x = df_subset[FEATURE_COLS].to_numpy()
y = df_subset[TARGET_COLS].to_numpy()

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

# Column normalization using training data only
mean = x_train.mean(axis=0)
sd   = x_train.std(axis=0)
sd   = np.where(sd == 0, 1, sd)
x_train = (x_train - mean) / sd
x_test  = (x_test  - mean) / sd

# Drop highly correlated columns using training data only
corr_matrix = np.corrcoef(x_train, rowvar=False)
corr_matrix_abs = np.abs(corr_matrix)

upper_triangle = np.triu(corr_matrix_abs, k=1)  # k=1 excludes the diagonal
cols_to_drop = np.any(upper_triangle > CORR_THRESHOLD, axis=0)

x_train = x_train[:, ~cols_to_drop]
x_test  = x_test[:,  ~cols_to_drop]

n_dropped = cols_to_drop.sum()
print(f"Columns dropped (threshold={CORR_THRESHOLD}): {n_dropped}")
print(f"Num. remaining features: {x_train.shape[1]}")

model = LinearRegression()
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_test  = model.predict(x_test)

rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))
rmse_test  = np.sqrt(mean_squared_error(y_test,  pred_test))
r2_train   = r2_score(y_train, pred_train)
r2_test    = r2_score(y_test,  pred_test)

print(f"\nTraining samples: {x_train.shape[0]}, Test samples: {x_test.shape[0]}")
print(f"\nTrain — RMSE: {rmse_train:.4f}  R²: {r2_train:.4f}")
print(f"Test  — RMSE: {rmse_test:.4f}  R²: {r2_test:.4f}")

Columns dropped (threshold=0.9): 72
Num. remaining features: 8

Training samples: 115, Test samples: 29

Train — RMSE: 0.2326  R²: 0.4206
Test  — RMSE: 0.2302  R²: 0.4862
